# Prepare clean paid-purchase history

## Goal

Convert the private sales export into one row per **customer + date + product**.

Only ordinary paid product sales (`ПРОДАЖА`) are retained. Gift rows (`ПОДАРОК`) are removed completely and do not affect labels, quantities, recency, reorder timing, or the product catalogue.

## 1. Setup

In [22]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_data_dir = project_root / "data" / "raw"

xlsx_raw_file = list(raw_data_dir.glob("*.xlsx"))[0]

if not xlsx_raw_file.exists():
    raise FileNotFoundError(f"Raw data file not found in {raw_data_dir}")
else:
    input_path = xlsx_raw_file

output_path = project_root / "data" / "interim" / "cleaned_purchases.csv"

## 2. Load, rename, and normalize columns

Customer and product codes are loaded as strings so pandas cannot interpret identifiers as numbers. Dates remain datetimes, and quantity remains numeric.

In [23]:
column_mapping = {
    "КлиентДляОплатыКод": "customer_id",
    "ТоварКод": "product_id",
    "Товар": "product_name",
    "Категория": "product_category",
    "БизнесЛиния": "business_line",
    "ДатаПродажи": "purchase_date",
    "Количество": "quantity",
    "Gen_ Bus_ Posting Group": "transaction_type",
    "Gen_ Prod_ Posting Group": "item_type",
}

identifier_dtypes = {
    "КлиентДляОплатыКод": "string",
    "ТоварКод": "string",
}

df = pd.read_excel(input_path, dtype=identifier_dtypes)

missing_columns = sorted(set(column_mapping) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected source columns: {missing_columns}")

df = df.rename(columns=column_mapping)

text_columns = [
    "customer_id",
    "product_id",
    "product_name",
    "product_category",
    "business_line",
    "transaction_type",
    "item_type",
]

for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

df["purchase_date"] = pd.to_datetime(df["purchase_date"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

print(f"Loaded {len(df):,} source rows.")
df.head(10)

Loaded 113,613 source rows.


,Межфирма,transaction_type,item_type,Компания,Дата,purchase_date,ФинСчет,ДокументДвижения,business_line,product_category,...,АкцияКод,Акция,ОписаниеАкции,УсловияОплаты,ТипОплаты,ТипКлиента,Услуга,УслугаНазвание,АдресДоставки,ILEDocType
0,0,ПРОДАЖА,ТОВАР,IG,2026-05-12,2026-05-12,281100,НОВА ПОШТА,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ПРЕДОПЛАТА,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Карачівське шосе, 44, Харків, Харківська облас...",1
1,0,ПРОДАЖА,ТОВАР,IG,2026-05-12,2026-05-12,281100,НОВА ПОШТА,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ПРЕДОПЛАТА,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Карачівське шосе, 44, Харків, Харківська облас...",1
2,0,ПРОДАЖА,ТОВАР,IG,2026-02-23,2026-02-23,281100,П Р С+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
3,0,ПРОДАЖА,ТОВАР,IG,2026-02-23,2026-02-23,281100,П Р С+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
4,0,ПРОДАЖА,ТОВАР,IG,2026-03-18,2026-03-18,281100,П РС+А 034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
5,0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,281100,П РС+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
6,0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,281100,П РС+А034555,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
7,0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,902100,П РС+А034555,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
8,0,ПРОДАЖА,ТОВАР,IG,2024-12-26,2024-12-26,281100,П Р С +У 9 4 3 5 04,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
9,0,ПРОДАЖА,ТОВАР,IG,2024-12-13,2024-12-13,281100,П Р С + У943504,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1


## 3. Keep positive paid product purchases

A retained row must identify a customer, date, and product; represent an actual product (`ТОВАР`); be an ordinary sale (`ПРОДАЖА`); and have a positive quantity. Gift rows and every other transaction type are excluded before aggregation.

In [24]:
required_text_columns = [
    "customer_id",
    "product_id",
    "product_name",
    "product_category",
    "business_line",
]
required_text = df[required_text_columns]
complete_required_values = (
    required_text.notna().all(axis=1)
    & required_text.ne("").all(axis=1)
    & df["purchase_date"].notna()
    & df["quantity"].notna()
)

purchase_mask = (
    complete_required_values
    & df["item_type"].eq("ТОВАР")
    & df["transaction_type"].eq("ПРОДАЖА")
    & df["quantity"].gt(0)
    & df["product_id"].str.startswith('ТОВ', na=False)
)

product_purchases = df.loc[purchase_mask].copy()

gift_product_rows = int(
    (
        df["item_type"].eq("ТОВАР")
        & df["transaction_type"].eq("ПОДАРОК")
        & df["quantity"].gt(0)
        & df["product_id"].str.startswith('ТОВ', na=True)
    ).sum()
)
assert product_purchases["transaction_type"].eq("ПРОДАЖА").all()
print(f"Excluded {gift_product_rows:,} positive gift-product rows.")
product_purchases.head(10)

Excluded 1,954 positive gift-product rows.


,Межфирма,transaction_type,item_type,Компания,Дата,purchase_date,ФинСчет,ДокументДвижения,business_line,product_category,...,АкцияКод,Акция,ОписаниеАкции,УсловияОплаты,ТипОплаты,ТипКлиента,Услуга,УслугаНазвание,АдресДоставки,ILEDocType
0,0,ПРОДАЖА,ТОВАР,IG,2026-05-12,2026-05-12,281100,НОВА ПОШТА,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ПРЕДОПЛАТА,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Карачівське шосе, 44, Харків, Харківська облас...",1
1,0,ПРОДАЖА,ТОВАР,IG,2026-05-12,2026-05-12,281100,НОВА ПОШТА,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ПРЕДОПЛАТА,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Карачівське шосе, 44, Харків, Харківська облас...",1
2,0,ПРОДАЖА,ТОВАР,IG,2026-02-23,2026-02-23,281100,П Р С+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
3,0,ПРОДАЖА,ТОВАР,IG,2026-02-23,2026-02-23,281100,П Р С+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
4,0,ПРОДАЖА,ТОВАР,IG,2026-03-18,2026-03-18,281100,П РС+А 034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
5,0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,281100,П РС+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
6,0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,281100,П РС+А034555,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
8,0,ПРОДАЖА,ТОВАР,IG,2024-12-26,2024-12-26,281100,П Р С +У 9 4 3 5 04,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
9,0,ПРОДАЖА,ТОВАР,IG,2024-12-13,2024-12-13,281100,П Р С + У943504,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1
10,0,ПРОДАЖА,ТОВАР,IG,2024-11-29,2024-11-29,281100,П Р С +У943504,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1


## 4. Unify volume-package variants and combine duplicates

Products whose names differ only by a trailing litre/millilitre package size are grouped by their complete normalized base name. Each group is represented by its smallest observed package: larger variants receive that package's product ID and product name.

For every matched row, `quantity` is converted into equivalent units of the smallest package: `original quantity × original package litres ÷ canonical package litres`. Products without a supported volume suffix are left unchanged. The resulting grouping key is `customer_id + purchase_date + canonical product_id`; repeated rows are then combined by summing the converted quantity.

In [25]:
import re

VOLUME_SUFFIX_PATTERN = (
    r",\s*(?P<package_amount>\d+(?:[.,]\d+)?)\s*"
    r"(?P<package_unit>ml|мл|l|л)\s*$"
)
volume_parts = product_purchases["product_name"].str.extract(
    VOLUME_SUFFIX_PATTERN,
    flags=re.IGNORECASE,
)

package_volume_litres = pd.to_numeric(
    volume_parts["package_amount"].str.replace(",", ".", regex=False),
    errors="coerce",
)
millilitre_rows = volume_parts["package_unit"].str.casefold().isin(["ml", "мл"])
package_volume_litres = package_volume_litres.mask(
    millilitre_rows,
    package_volume_litres / 1_000,
)
volume_variant_rows = package_volume_litres.gt(0)

normalized_product_base = (
    product_purchases["product_name"]
    .str.replace(VOLUME_SUFFIX_PATTERN, "", regex=True, flags=re.IGNORECASE)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.casefold()
)

volume_catalog = (
    product_purchases.loc[
        volume_variant_rows,
        ["product_id", "product_name"],
    ]
    .assign(
        normalized_product_base=normalized_product_base.loc[volume_variant_rows],
        package_volume_litres=package_volume_litres.loc[volume_variant_rows],
    )
    .drop_duplicates()
    .sort_values(
        ["normalized_product_base", "package_volume_litres", "product_id"],
        kind="stable",
    )
)
canonical_products = (
    volume_catalog
    .drop_duplicates("normalized_product_base", keep="first")
    .set_index("normalized_product_base")
)

product_count_before_unification = product_purchases["product_id"].nunique()
quantity_before_unification = product_purchases["quantity"].sum()
canonical_product_ids = normalized_product_base.map(canonical_products["product_id"])
canonical_package_volume_litres = normalized_product_base.map(
    canonical_products["package_volume_litres"]
)
package_conversion_factors = (
    package_volume_litres / canonical_package_volume_litres
)
remapped_volume_rows = int(
    product_purchases.loc[volume_variant_rows, "product_id"]
    .ne(canonical_product_ids.loc[volume_variant_rows])
    .sum()
)

expected_quantity = product_purchases["quantity"].copy()
expected_quantity.loc[volume_variant_rows] = (
    expected_quantity.loc[volume_variant_rows]
    * package_conversion_factors.loc[volume_variant_rows]
)
product_purchases.loc[volume_variant_rows, "quantity"] = (
    expected_quantity.loc[volume_variant_rows].to_numpy()
)

canonical_columns = [
    "product_id",
    "product_name",
]
for column in canonical_columns:
    product_purchases.loc[volume_variant_rows, column] = (
        normalized_product_base.loc[volume_variant_rows]
        .map(canonical_products[column])
        .to_numpy()
    )

quantity_after_unification = product_purchases["quantity"].sum()
product_count_after_unification = product_purchases["product_id"].nunique()
pd.testing.assert_series_equal(
    product_purchases["quantity"],
    expected_quantity,
    check_names=False,
)

grouping_columns = ["customer_id", "purchase_date", "product_id"]

cleaned_purchases = (
    product_purchases
    .groupby(
        grouping_columns,
        as_index=False,
        sort=False,
    )
    .agg(
        quantity=("quantity", "sum"),
        product_category=("product_category", "first"),
        business_line=("business_line", "first"),
        product_name=("product_name", "first"),
    )
    .sort_values(grouping_columns)
    .reset_index(drop=True)
)

unification_summary = pd.Series({
    "volume_rows_detected": int(volume_variant_rows.sum()),
    "volume_rows_remapped": remapped_volume_rows,
    "products_before_unification": product_count_before_unification,
    "products_after_unification": product_count_after_unification,
    "quantity_before_unification": quantity_before_unification,
    "quantity_after_unification": quantity_after_unification,
    "rows_after_aggregation": len(cleaned_purchases),
})
unification_summary

volume_rows_detected           5.876600e+04
volume_rows_remapped           3.860100e+04
products_before_unification    9.270000e+02
products_after_unification     7.210000e+02
quantity_before_unification    2.167768e+07
quantity_after_unification     2.717967e+08
rows_after_aggregation         6.951700e+04
dtype: float64

## 5. Validate and save

The checks stop the notebook if duplicate keys, non-positive purchases, gift-related fields, or missing numeric values survive. The validated table is then saved as CSV with dates written as `YYYY-MM-DD`.

In [26]:
duplicate_count = int(cleaned_purchases.duplicated(grouping_columns).sum())
non_positive_purchase_count = int(cleaned_purchases["quantity"].le(0).sum())
quantity_difference = abs(
    cleaned_purchases["quantity"].sum() - quantity_after_unification
)
quantity_tolerance = 1e-9 * max(1.0, abs(quantity_after_unification))

assert duplicate_count == 0, f"Found {duplicate_count} duplicate grouping keys."
assert non_positive_purchase_count == 0, (
    f"Found {non_positive_purchase_count} non-positive purchases."
)
assert quantity_difference <= quantity_tolerance, (
    f"Quantity changed by {quantity_difference} during aggregation."
)
assert cleaned_purchases.notna().all().all()
assert cleaned_purchases[required_text_columns].ne("").all().all()
assert {
    "gift_quantity",
    "received_quantity",
    "paid_quantity",
}.isdisjoint(cleaned_purchases.columns)

output_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_purchases.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d",
)

saved_purchases = pd.read_csv(
    output_path,
    dtype={
        "customer_id": "string",
        "product_id": "string",
        "product_name": "string",
        "product_category": "string",
        "business_line": "string",
    },
    parse_dates=["purchase_date"],
)
assert len(saved_purchases) == len(cleaned_purchases)
assert list(saved_purchases.columns) == list(cleaned_purchases.columns)

validation_summary = pd.Series({
    "source_rows": len(df),
    "eligible_paid_purchase_rows": len(product_purchases),
    "excluded_positive_gift_rows": gift_product_rows,
    "cleaned_customer_date_product_rows": len(cleaned_purchases),
    "rows_combined_after_unification": len(product_purchases) - len(cleaned_purchases),
    "remaining_duplicate_keys": duplicate_count,
    "customers": cleaned_purchases["customer_id"].nunique(),
    "products": cleaned_purchases["product_id"].nunique(),
})

print(f"Saved validated data to {output_path}")
validation_summary

Saved validated data to /Users/kalyma123/programming/reccomendation algorithm/data/interim/cleaned_purchases.csv


source_rows                           113613
eligible_paid_purchase_rows            76797
excluded_positive_gift_rows             1954
cleaned_customer_date_product_rows     69517
rows_combined_after_unification         7280
remaining_duplicate_keys                   0
customers                               3711
products                                 721
dtype: int64